## Lab 04
### Luis Eduardo Gonzalez

In [1]:
from spark_utils import SparkUtils
from pyspark.sql.functions import col, get_json_object

spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Lab 04"
su = SparkUtils(app_name, spark_url)
spark = su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 17:29:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
agencies_schema = SparkUtils.generate_schema([
    ("agency_id", "int"),
    ("agency_info", "string")
])

agencies_data = [
    (1, '{"agency_name":"NYC Rentals","city":"New York"}'),
    (2, '{"agency_name":"LA Car Rental","city":"Londres"}'),
    (3, '{"agency_name":"Zapopan Auto","city":"Zapopan"}'),
    (4, '{"agency_name":"SF Cars","city":"San Francisco"}'),
    (5, '{"agency_name":"Mexico Cars","city":"Mexico City"}')
]

agencies_df = spark.createDataFrame(
    agencies_data,
    agencies_schema
)

In [3]:
brands_schema = SparkUtils.generate_schema([
    ("brand_id", "int"),
    ("brand_info", "string")
])

brands_data = [
    (1, '{"brand_name":"Mercedes-Benz","country":"Germany"}'),
    (2, '{"brand_name":"BMW","country":"Germany"}'),
    (3, '{"brand_name":"Audi","country":"Senegal"}'),
    (4, '{"brand_name":"Ford","country":"Tuvalu"}'),
    (5, '{"brand_name":"BYD","country":"Italy"}')
]

brands_df = spark.createDataFrame(brands_data, brands_schema)

In [4]:
cars_schema = SparkUtils.generate_schema([
    ("car_id", "int"),
    ("car_info", "string")
])

cars_data = [
    (1, '{"car_name":"Salazar Ltd Model 6","brand_id":1}'),
    (2, '{"car_name":"Harris, Lloyd and Payne Model 4","brand_id":2}'),
    (3, '{"car_name":"Alvarez-Davis Model 6","brand_id":3}'),
    (4, '{"car_name":"Lopez and Sons Model 3","brand_id":4}'),
    (5, '{"car_name":"Levy Group Model 8","brand_id":5}')
]

cars_df = spark.createDataFrame(cars_data, cars_schema)

In [5]:
customers_schema = SparkUtils.generate_schema([
    ("customer_id", "int"),
    ("customer_info", "string")
])

customers_data = [
    (42, '{"customer_name":"Sara Anderson"}'),
    (146, '{"customer_name":"Calvin Walker"}'),
    (143, '{"customer_name":"Shawn Tran"}'),
    (90, '{"customer_name":"Edward Mccarthy"}'),
    (115, '{"customer_name":"Antonio Haynes"}')
]

customers_df = spark.createDataFrame(customers_data, customers_schema)

In [6]:
rentals_schema = SparkUtils.generate_schema([
    ("rental_id", "int"),
    ("rental_info", "string")
])

rentals_data = [
    (12740, '{"car_id":1,"customer_id":42,"agency_id":1}'),
    (12741, '{"car_id":2,"customer_id":146,"agency_id":2}'),
    (12742, '{"car_id":3,"customer_id":143,"agency_id":3}'),
    (12743, '{"car_id":4,"customer_id":90,"agency_id":4}'),
    (12744, '{"car_id":5,"customer_id":115,"agency_id":3}')
]

rentals_df = spark.createDataFrame(rentals_data, rentals_schema)

In [7]:
agencies_df = agencies_df \
    .withColumn(
        "agency_name",
        get_json_object(col("agency_info"), "$.agency_name")
    ) \
    .withColumn(
        "agency_city",
        get_json_object(col("agency_info"), "$.city")
    )

In [8]:
brands_df = brands_df \
    .withColumn(
        "brand_name",
        get_json_object(col("brand_info"), "$.brand_name")
    ) \
    .withColumn(
        "brand_country",
        get_json_object(col("brand_info"), "$.country")
    )

In [9]:
cars_df = cars_df \
    .withColumn(
        "car_name",
        get_json_object(col("car_info"), "$.car_name")
    ) \
    .withColumn(
        "brand_id",
        get_json_object(col("car_info"), "$.brand_id").cast("int")
    )

In [10]:
customers_df = customers_df \
    .withColumn(
        "customer_name",
        get_json_object(col("customer_info"), "$.customer_name")
    )

In [11]:
rentals_df = rentals_df \
    .withColumn(
        "car_id",
        get_json_object(col("rental_info"), "$.car_id").cast("int")
    ) \
    .withColumn(
        "customer_id",
        get_json_object(col("rental_info"), "$.customer_id").cast("int")
    ) \
    .withColumn(
        "agency_id",
        get_json_object(col("rental_info"), "$.agency_id").cast("int")
    )

In [12]:
final_df = rentals_df \
    .join(cars_df, "car_id") \
    .join(customers_df, "customer_id") \
    .join(agencies_df, "agency_id") \
    .select(
        col("rental_id"),
        col("car_name"),
        col("agency_name"),
        col("customer_name")
    ) \
    .orderBy(col("rental_id"))

In [13]:
final_df.show(truncate=False)

+---------+-------------------------------+-------------+---------------+
|rental_id|car_name                       |agency_name  |customer_name  |
+---------+-------------------------------+-------------+---------------+
|12740    |Salazar Ltd Model 6            |NYC Rentals  |Sara Anderson  |
|12741    |Harris, Lloyd and Payne Model 4|LA Car Rental|Calvin Walker  |
|12742    |Alvarez-Davis Model 6          |Zapopan Auto |Shawn Tran     |
|12743    |Lopez and Sons Model 3         |SF Cars      |Edward Mccarthy|
|12744    |Levy Group Model 8             |Zapopan Auto |Antonio Haynes |
+---------+-------------------------------+-------------+---------------+

